# Feature Engineering

This notebook transforms credit-level history into a client-level feature matrix for default prediction.

The feature engineering strategy is based on the findings from exploratory data analysis:

- each client may have multiple credit products;
- `rn` preserves the chronological order of credit products;
- `pre_*` features are binarized and their numerical codes should not be treated as continuous values;
- `enc_*` features represent encoded categorical information;
- delinquency history differs substantially between default and non-default clients;
- payment history contains sequential information that should be preserved where possible.

The goal is to construct interpretable client-level features without imposing artificial numerical ordering on encoded categories.

## 1. Importing libraries and path to data

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm import tqdm

pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "train_data"
TARGET_PATH = PROJECT_ROOT / "data" / "train_target.csv"

## 2. Loading Credit History

The credit history is stored across multiple Parquet partitions. We load the partitions and combine them before client-level feature engineering to ensure that each client's complete credit history is available during aggregation.

In [3]:
def load_parquet_partitions(data_dir: Path,
                            num_parts: int | None = None,
                            columns: list[str] | None = None) -> pd.DataFrame:
    partition_path = sorted(data_dir.glob("train_data_*.pq"),
                            key= lambda path: int(re.search(r"\d+", path.stem).group()))

    if num_parts is not None:
        partition_path = partition_path[:num_parts]

    frames = []

    for path in tqdm(partition_path, desc="Reading partitions"):
        frame = pd.read_parquet(path, columns=columns)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

In [4]:
df = load_parquet_partitions(DATA_DIR,
                             num_parts=2)
print(f"shape: {df.shape}")
print(f"Number of clients: {df['id'].nunique():,}")

df.head()

Reading partitions: 100%|██████████| 2/2 [00:00<00:00,  5.31it/s]


shape: (4082029, 61)
Number of clients: 500,000


,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,3,0,2,11,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,3,0,2,11,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,5,0,2,8,6,16,5,4,8,1,1,1,1,1,15,2,17,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12,2,3,0,2,4,6,16,5,4,8,0,1,1,1,1,16,2,17,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10,2,3,0,2,4,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0


### Partition Structure Check

Before processing partitions independently, we verify whether the same client can appear in multiple partitions. If client histories are split across files, aggregation must be performed only after combining the relevant partitions.

In [5]:
part_0_ids = pd.read_parquet(DATA_DIR / "train_data_0.pq",
                             columns=["id"])["id"].unique()

part_1_ids = pd.read_parquet(DATA_DIR / "train_data_1.pq",
                             columns=["id"])["id"].unique()

common_clients = np.intersect1d(part_0_ids, part_1_ids)

print(f"Clients in partition 0: {len(part_0_ids):,}")
print(f"Clients in partition 1: {len(part_1_ids):,}")
print(f"Clients appearing in both: {len(common_clients):,}")

Clients in partition 0: 250,000
Clients in partition 1: 250,000
Clients appearing in both: 0


In [6]:
partition_path = sorted(DATA_DIR.glob("train_data_*.pq"),
                        key=lambda path: int(re.search(r"\d+", path.stem).group()))
seen_ids = set()
overlapping_ids = set()

for path in tqdm(partition_path, desc="Checking partitions"):
    ids = set(pd.read_parquet(path, columns=["id"])["id"].unique())

    overlapping_ids.update(seen_ids.intersection(ids))
    seen_ids.update(ids)

print(f"Partitions checked: {len(partition_path)}")
print(f"Unique clients: {len(seen_ids):,}")
print(f"Clients appearing in multiple partitions: {len(overlapping_ids):,}")

Checking partitions: 100%|██████████| 12/12 [00:00<00:00, 15.06it/s]

Partitions checked: 12
Unique clients: 3,000,000
Clients appearing in multiple partitions: 0


## 3. Client-Level Feature Engineering

The raw dataset contains multiple credit records for each client, while the prediction target is defined at the client level.

Since each client's complete credit history is stored within a single partition, the partitions can be processed independently and the resulting client-level datasets can be combined afterwards.

We begin with basic features describing the size and structure of each client's credit history.

### 3.1 Basic Credit History Features

The first group of features describes the number of credit products in each client's history.

In [7]:
def create_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    basic_features = (df.groupby("id").agg(credit_count=("rn", "count")))
    return basic_features

In [24]:
basic_features = create_basic_features(df)

print(f"Shape: {basic_features.shape}")

basic_features.head()

Shape: (500000, 1)


,credit_count
id,
0,10
1,14
2,3
3,15
4,1


### 3.2 Delinquency Features

The dataset contains binary indicators showing whether a credit record has no delinquency events of a given duration.

For each delinquency category, we construct two client-level features:

- `ever_*` — indicates whether the client has experienced at least one delinquency event of this duration across their credit history;
- `share_*` — represents the proportion of the client's credit records that contain at least one delinquency event of this duration.

These features capture both the presence and the frequency of delinquency in the client's credit history.

In [8]:
delinquency_cols = {"is_zero_loans5": "loans5",
                    "is_zero_loans530": "loans530",
                    "is_zero_loans3060": "loans3060",
                    "is_zero_loans6090": "loans6090",
                    "is_zero_loans90": "loans90"
}

In [9]:
def create_delinquency_features(df: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame(index=df["id"].unique())
    grouped = df.groupby("id")

    for col, name in delinquency_cols.items():
        features[f"ever_{name}"] = 1 - grouped[col].min()
        features[f"share_{name}"] = 1- grouped[col].mean()

    features.index.name = "id"

    return features

In [26]:
delinquency_features = create_delinquency_features(df)

print(f"Shape: {delinquency_features.shape}")

delinquency_features.head()

Shape: (500000, 10)


,ever_loans5,share_loans5,ever_loans530,share_loans530,ever_loans3060,share_loans3060,ever_loans6090,share_loans6090,ever_loans90,share_loans90
id,,,,,,,,,,
0,1,0.100000,0,0.000000,0,0.000000,0,0.000000,0,0.000000
1,1,0.142857,1,0.285714,1,0.142857,1,0.142857,1,0.214286
2,0,0.000000,1,0.333333,1,0.333333,1,0.333333,0,0.000000
3,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000
4,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000


### 3.3 Chronological Credit Features

The `rn` variable preserves the chronological order of credit products within each client's history: higher values correspond to more recently opened credit products.

To preserve this temporal information after client-level aggregation, we extract selected characteristics of the earliest and most recent credit records.

Since `pre_*` and `enc_*` variables contain arbitrarily encoded categories or intervals, their values are preserved as categories and are not subtracted or otherwise treated as ordered numerical quantities.

In [11]:
chronological_cols = [
    # Delinquency indicators
    "is_zero_loans5",
    "is_zero_loans530",
    "is_zero_loans3060",
    "is_zero_loans6090",
    "is_zero_loans90",

    # Utilization indicators
    "is_zero_util",
    "is_zero_over2limit",
    "is_zero_maxover2limit",

    # Credit characteristics
    "pre_loans_credit_limit",
    "pre_loans_outstanding",
    "pre_util",

    # Credit categories
    "enc_loans_credit_type",
    "enc_loans_credit_status",
]

In [12]:
def create_chronological_features(df: pd.DataFrame) -> pd.DataFrame:
    sorted_df = df.sort_values(["id", "rn"])

    first_credit = (sorted_df.groupby("id")[chronological_cols].first().add_suffix("_first"))
    last_credit = (sorted_df.groupby("id")[chronological_cols].last().add_suffix("_last"))

    features = first_credit.join(last_credit)

    return features

In [13]:
chronological_features = create_chronological_features(df)
print(f"Shape: {chronological_features.shape}")

chronological_features.head()

Shape: (500000, 26)


,is_zero_loans5_first,is_zero_loans530_first,is_zero_loans3060_first,is_zero_loans6090_first,is_zero_loans90_first,is_zero_util_first,is_zero_over2limit_first,is_zero_maxover2limit_first,pre_loans_credit_limit_first,pre_loans_outstanding_first,pre_util_first,enc_loans_credit_type_first,enc_loans_credit_status_first,is_zero_loans5_last,is_zero_loans530_last,is_zero_loans3060_last,is_zero_loans6090_last,is_zero_loans90_last,is_zero_util_last,is_zero_over2limit_last,is_zero_maxover2limit_last,pre_loans_credit_limit_last,pre_loans_outstanding_last,pre_util_last,enc_loans_credit_type_last,enc_loans_credit_status_last
id,,,,,,,,,,,,,,,,,,,,,,,,,,
0,1,1,1,1,1,1,1,1,11,3,16,4,3,1,1,1,1,1,0,1,1,16,2,15,4,2
1,1,1,0,0,0,0,0,0,1,4,2,4,3,1,1,1,1,1,1,1,1,11,3,16,3,2
2,1,0,0,0,1,0,1,0,1,2,6,3,2,1,1,1,1,1,0,1,1,1,4,3,3,2
3,1,1,1,1,1,1,1,1,13,3,16,4,2,1,1,1,1,1,0,1,1,17,2,6,4,2
4,1,1,1,1,1,1,1,1,12,3,16,3,2,1,1,1,1,1,1,1,1,12,3,16,3,2


### 3.4 Categorical Credit History Features

Encoded credit characteristics represent categorical information and should not be treated as ordered numerical values.

To capture the diversity of a client's credit history, we calculate the number of unique categories observed across their credit records.

These features describe how varied the client's credit products, account types, and credit statuses have been without imposing an artificial numerical ordering on the encoded categories.

In [14]:
categorical_cols = [
    "enc_loans_account_cur",
    "enc_loans_credit_type",
    "enc_loans_account_holder_type",
    "enc_loans_credit_status",
]

In [15]:
def create_categorical_features(df: pd.DataFrame) -> pd.DataFrame:
    features = (df.groupby("id")[categorical_cols].nunique().add_suffix("_nunique"))

    return features

In [16]:
categorical_features = create_categorical_features(df)

print(f"Shape: {categorical_features.shape}")

categorical_features.head()

Shape: (500000, 4)


,enc_loans_account_cur_nunique,enc_loans_credit_type_nunique,enc_loans_account_holder_type_nunique,enc_loans_credit_status_nunique
id,,,,
0,1,3,1,2
1,1,3,1,2
2,1,2,1,2
3,1,4,1,2
4,1,1,1,1


### 3.5 Binarized Credit Characteristics

The `pre_*` variables represent binarized credit characteristics. Their numerical codes identify intervals of the original variables but should not be interpreted as continuous measurements or used to infer magnitude.

To preserve information from these features without imposing artificial numerical ordering, we calculate the number of unique encoded intervals observed in each client's credit history.

This captures the diversity of credit characteristics across a client's credit products.

In [17]:
pre_cols = [col for col in df.columns if col.startswith("pre_")]

print(f"Number of pre_* features: {len(pre_cols)}")
pre_cols

Number of pre_* features: 20


['pre_since_opened',
 'pre_since_confirmed',
 'pre_pterm',
 'pre_fterm',
 'pre_till_pclose',
 'pre_till_fclose',
 'pre_loans_credit_limit',
 'pre_loans_next_pay_summ',
 'pre_loans_outstanding',
 'pre_loans_total_overdue',
 'pre_loans_max_overdue_sum',
 'pre_loans_credit_cost_rate',
 'pre_loans5',
 'pre_loans530',
 'pre_loans3060',
 'pre_loans6090',
 'pre_loans90',
 'pre_util',
 'pre_over2limit',
 'pre_maxover2limit']

In [18]:
def create_binarized_features(df: pd.DataFrame) -> pd.DataFrame:
    features = (df.groupby("id")[pre_cols].nunique().add_suffix("_nunique"))

    return features

In [19]:
binarized_features = create_binarized_features(df)

print(f"Shape: {binarized_features.shape}")

binarized_features.head()

Shape: (500000, 20)


,pre_since_opened_nunique,pre_since_confirmed_nunique,pre_pterm_nunique,pre_fterm_nunique,pre_till_pclose_nunique,pre_till_fclose_nunique,pre_loans_credit_limit_nunique,pre_loans_next_pay_summ_nunique,pre_loans_outstanding_nunique,pre_loans_total_overdue_nunique,pre_loans_max_overdue_sum_nunique,pre_loans_credit_cost_rate_nunique,pre_loans5_nunique,pre_loans530_nunique,pre_loans3060_nunique,pre_loans6090_nunique,pre_loans90_nunique,pre_util_nunique,pre_over2limit_nunique,pre_maxover2limit_nunique
id,,,,,,,,,,,,,,,,,,,,
0,7,4,7,7,6,6,8,5,4,1,1,5,1,1,1,1,1,4,2,2
1,9,8,9,10,7,8,10,5,3,1,3,4,1,1,1,1,1,5,3,3
2,3,2,2,2,2,2,2,2,3,1,2,3,1,2,1,1,1,3,1,2
3,8,9,9,9,7,7,11,5,4,1,1,8,1,1,1,1,1,8,2,2
4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


### 3.6 Payment History Features

The `enc_paym_*` variables represent encoded monthly payment statuses. Each variable contains arbitrary numerical category codes, so the codes are not interpreted as ordered values.

The observed code sets also differ across payment-history variables. Therefore, we avoid comparing numerical codes directly between different `enc_paym_*` periods or interpreting transitions between them as improvements or deteriorations.

Instead, each payment-history variable is treated independently. For every client, we calculate the number of unique encoded statuses observed across their credit records for each payment-history variable.

In [20]:
payment_cols = sorted([col for col in df.columns if col.startswith("enc_paym_")],
                      key=lambda col: int(col.split("_")[-1])
)

print(f"Number of payment history features: {len(payment_cols)}")
payment_cols

Number of payment history features: 25


['enc_paym_0',
 'enc_paym_1',
 'enc_paym_2',
 'enc_paym_3',
 'enc_paym_4',
 'enc_paym_5',
 'enc_paym_6',
 'enc_paym_7',
 'enc_paym_8',
 'enc_paym_9',
 'enc_paym_10',
 'enc_paym_11',
 'enc_paym_12',
 'enc_paym_13',
 'enc_paym_14',
 'enc_paym_15',
 'enc_paym_16',
 'enc_paym_17',
 'enc_paym_18',
 'enc_paym_19',
 'enc_paym_20',
 'enc_paym_21',
 'enc_paym_22',
 'enc_paym_23',
 'enc_paym_24']

In [21]:
def create_payment_features(df: pd.DataFrame) -> pd.DataFrame:
    features = (df.groupby("id")[payment_cols].nunique().add_suffix("_nunique"))

    return features

In [22]:
payment_features = create_payment_features(df)

print(f"Shape: {payment_features.shape}")

payment_features.head()

Shape: (500000, 25)


,enc_paym_0_nunique,enc_paym_1_nunique,enc_paym_2_nunique,enc_paym_3_nunique,enc_paym_4_nunique,enc_paym_5_nunique,enc_paym_6_nunique,enc_paym_7_nunique,enc_paym_8_nunique,enc_paym_9_nunique,enc_paym_10_nunique,enc_paym_11_nunique,enc_paym_12_nunique,enc_paym_13_nunique,enc_paym_14_nunique,enc_paym_15_nunique,enc_paym_16_nunique,enc_paym_17_nunique,enc_paym_18_nunique,enc_paym_19_nunique,enc_paym_20_nunique,enc_paym_21_nunique,enc_paym_22_nunique,enc_paym_23_nunique,enc_paym_24_nunique
id,,,,,,,,,,,,,,,,,,,,,,,,,
0,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1
1,3,3,3,2,2,2,3,3,2,3,2,2,2,2,2,3,2,2,2,2,2,2,3,2,2
2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2,1
3,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


## 4. Building the Client-Level Feature Matrix

The feature groups created above capture different aspects of each client's credit history.

We now combine them into a single client-level feature matrix, where each row represents one client and each column represents an engineered feature.

In [27]:
client_features = (basic_features.join(delinquency_features)
                                 .join(chronological_features)
                                 .join(categorical_features)
                                 .join(binarized_features)
                                 .join(payment_features))

print(f"Shape: {client_features.shape}")

client_features.head()

Shape: (500000, 86)


,credit_count,ever_loans5,share_loans5,ever_loans530,share_loans530,ever_loans3060,share_loans3060,ever_loans6090,share_loans6090,ever_loans90,share_loans90,is_zero_loans5_first,is_zero_loans530_first,is_zero_loans3060_first,is_zero_loans6090_first,is_zero_loans90_first,is_zero_util_first,is_zero_over2limit_first,is_zero_maxover2limit_first,pre_loans_credit_limit_first,pre_loans_outstanding_first,pre_util_first,enc_loans_credit_type_first,enc_loans_credit_status_first,is_zero_loans5_last,is_zero_loans530_last,is_zero_loans3060_last,is_zero_loans6090_last,is_zero_loans90_last,is_zero_util_last,is_zero_over2limit_last,is_zero_maxover2limit_last,pre_loans_credit_limit_last,pre_loans_outstanding_last,pre_util_last,enc_loans_credit_type_last,enc_loans_credit_status_last,enc_loans_account_cur_nunique,enc_loans_credit_type_nunique,enc_loans_account_holder_type_nunique,enc_loans_credit_status_nunique,pre_since_opened_nunique,pre_since_confirmed_nunique,pre_pterm_nunique,pre_fterm_nunique,pre_till_pclose_nunique,pre_till_fclose_nunique,pre_loans_credit_limit_nunique,pre_loans_next_pay_summ_nunique,pre_loans_outstanding_nunique,pre_loans_total_overdue_nunique,pre_loans_max_overdue_sum_nunique,pre_loans_credit_cost_rate_nunique,pre_loans5_nunique,pre_loans530_nunique,pre_loans3060_nunique,pre_loans6090_nunique,pre_loans90_nunique,pre_util_nunique,pre_over2limit_nunique,pre_maxover2limit_nunique,enc_paym_0_nunique,enc_paym_1_nunique,enc_paym_2_nunique,enc_paym_3_nunique,enc_paym_4_nunique,enc_paym_5_nunique,enc_paym_6_nunique,enc_paym_7_nunique,enc_paym_8_nunique,enc_paym_9_nunique,enc_paym_10_nunique,enc_paym_11_nunique,enc_paym_12_nunique,enc_paym_13_nunique,enc_paym_14_nunique,enc_paym_15_nunique,enc_paym_16_nunique,enc_paym_17_nunique,enc_paym_18_nunique,enc_paym_19_nunique,enc_paym_20_nunique,enc_paym_21_nunique,enc_paym_22_nunique,enc_paym_23_nunique,enc_paym_24_nunique
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,10,1,0.100000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,1,1,1,1,1,1,1,1,11,3,16,4,3,1,1,1,1,1,0,1,1,16,2,15,4,2,1,3,1,2,7,4,7,7,6,6,8,5,4,1,1,5,1,1,1,1,1,4,2,2,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1
1,14,1,0.142857,1,0.285714,1,0.142857,1,0.142857,1,0.214286,1,1,0,0,0,0,0,0,1,4,2,4,3,1,1,1,1,1,1,1,1,11,3,16,3,2,1,3,1,2,9,8,9,10,7,8,10,5,3,1,3,4,1,1,1,1,1,5,3,3,3,3,3,2,2,2,3,3,2,3,2,2,2,2,2,3,2,2,2,2,2,2,3,2,2
2,3,0,0.000000,1,0.333333,1,0.333333,1,0.333333,0,0.000000,1,0,0,0,1,0,1,0,1,2,6,3,2,1,1,1,1,1,0,1,1,1,4,3,3,2,1,2,1,2,3,2,2,2,2,2,2,2,3,1,2,3,1,2,1,1,1,3,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2,1
3,15,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,1,1,1,1,1,1,1,1,13,3,16,4,2,1,1,1,1,1,0,1,1,17,2,6,4,2,1,4,1,2,8,9,9,9,7,7,11,5,4,1,1,8,1,1,1,1,1,8,2,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
4,1,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,1,1,1,1,1,1,1,1,12,3,16,3,2,1,1,1,1,1,1,1,1,12,3,16,3,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [28]:
print(f"Clients: {client_features.shape[0]:,}")
print(f"Features: {client_features.shape[1]}")
print(f"Duplicate client IDs: {client_features.index.duplicated().sum():,}")
print(f"Missing values: {client_features.isna().sum().sum():,}")

Clients: 500,000
Features: 86
Duplicate client IDs: 0
Missing values: 0


## 5. Processing All Data Partitions

During development, the feature engineering pipeline was tested on two data partitions.

Since each client appears in exactly one partition, the partitions can be processed independently. This allows the complete dataset to be transformed without loading all credit-level records into memory at once.

For each partition, we create the same client-level features and then concatenate the resulting feature matrices.

In [29]:
def create_client_features(df: pd.DataFrame) -> pd.DataFrame:
    basic_features = create_basic_features(df)
    delinquency_features = create_delinquency_features(df)
    chronological_features = create_chronological_features(df)
    categorical_features = create_categorical_features(df)
    binarized_features = create_binarized_features(df)
    payment_features = create_payment_features(df)

    client_features = (basic_features.join(delinquency_features)
                                     .join(chronological_features)
                                     .join(categorical_features)
                                     .join(binarized_features)
                                     .join(payment_features))

    return client_features

In [31]:
partition_paths = sorted(DATA_DIR.glob("train_data_*.pq"),
                         key=lambda path: int(re.search(r"\d+", path.stem).group()))

feature_frames = []

for path in tqdm(partition_path, desc="Processing partitions"):
    partition_df = pd.read_parquet(path)

    partition_features = create_client_features(partition_df)
    feature_frames.append(partition_features)

    del partition_df
    del partition_features

all_client_features = pd.concat(feature_frames)
print(f"Shape: {all_client_features.shape}")

Processing partitions: 100%|██████████| 12/12 [00:40<00:00,  3.36s/it]


Shape: (3000000, 86)


## 6. Adding the Target Variable

The target variable is stored separately from the credit history data.

We load the target dataset and merge it with the engineered client-level features using the client identifier `id`.

In [33]:
target = pd.read_csv(TARGET_PATH)

print(f"Shape: {target.shape}")

target.head()

Shape: (3000000, 2)


,id,flag
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [35]:
final_dataset = all_client_features.join(target.set_index("id"))
print(f"Shape: {final_dataset.shape}")
final_dataset.head()

Shape: (3000000, 87)


,credit_count,ever_loans5,share_loans5,ever_loans530,share_loans530,ever_loans3060,share_loans3060,ever_loans6090,share_loans6090,ever_loans90,share_loans90,is_zero_loans5_first,is_zero_loans530_first,is_zero_loans3060_first,is_zero_loans6090_first,is_zero_loans90_first,is_zero_util_first,is_zero_over2limit_first,is_zero_maxover2limit_first,pre_loans_credit_limit_first,pre_loans_outstanding_first,pre_util_first,enc_loans_credit_type_first,enc_loans_credit_status_first,is_zero_loans5_last,is_zero_loans530_last,is_zero_loans3060_last,is_zero_loans6090_last,is_zero_loans90_last,is_zero_util_last,is_zero_over2limit_last,is_zero_maxover2limit_last,pre_loans_credit_limit_last,pre_loans_outstanding_last,pre_util_last,enc_loans_credit_type_last,enc_loans_credit_status_last,enc_loans_account_cur_nunique,enc_loans_credit_type_nunique,enc_loans_account_holder_type_nunique,enc_loans_credit_status_nunique,pre_since_opened_nunique,pre_since_confirmed_nunique,pre_pterm_nunique,pre_fterm_nunique,pre_till_pclose_nunique,pre_till_fclose_nunique,pre_loans_credit_limit_nunique,pre_loans_next_pay_summ_nunique,pre_loans_outstanding_nunique,pre_loans_total_overdue_nunique,pre_loans_max_overdue_sum_nunique,pre_loans_credit_cost_rate_nunique,pre_loans5_nunique,pre_loans530_nunique,pre_loans3060_nunique,pre_loans6090_nunique,pre_loans90_nunique,pre_util_nunique,pre_over2limit_nunique,pre_maxover2limit_nunique,enc_paym_0_nunique,enc_paym_1_nunique,enc_paym_2_nunique,enc_paym_3_nunique,enc_paym_4_nunique,enc_paym_5_nunique,enc_paym_6_nunique,enc_paym_7_nunique,enc_paym_8_nunique,enc_paym_9_nunique,enc_paym_10_nunique,enc_paym_11_nunique,enc_paym_12_nunique,enc_paym_13_nunique,enc_paym_14_nunique,enc_paym_15_nunique,enc_paym_16_nunique,enc_paym_17_nunique,enc_paym_18_nunique,enc_paym_19_nunique,enc_paym_20_nunique,enc_paym_21_nunique,enc_paym_22_nunique,enc_paym_23_nunique,enc_paym_24_nunique,flag
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,10,1,0.100000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,1,1,1,1,1,1,1,1,11,3,16,4,3,1,1,1,1,1,0,1,1,16,2,15,4,2,1,3,1,2,7,4,7,7,6,6,8,5,4,1,1,5,1,1,1,1,1,4,2,2,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,0
1,14,1,0.142857,1,0.285714,1,0.142857,1,0.142857,1,0.214286,1,1,0,0,0,0,0,0,1,4,2,4,3,1,1,1,1,1,1,1,1,11,3,16,3,2,1,3,1,2,9,8,9,10,7,8,10,5,3,1,3,4,1,1,1,1,1,5,3,3,3,3,3,2,2,2,3,3,2,3,2,2,2,2,2,3,2,2,2,2,2,2,3,2,2,0
2,3,0,0.000000,1,0.333333,1,0.333333,1,0.333333,0,0.000000,1,0,0,0,1,0,1,0,1,2,6,3,2,1,1,1,1,1,0,1,1,1,4,3,3,2,1,2,1,2,3,2,2,2,2,2,2,2,3,1,2,3,1,2,1,1,1,3,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2,1,0
3,15,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,1,1,1,1,1,1,1,1,13,3,16,4,2,1,1,1,1,1,0,1,1,17,2,6,4,2,1,4,1,2,8,9,9,9,7,7,11,5,4,1,1,8,1,1,1,1,1,8,2,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,0
4,1,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,1,1,1,1,1,1,1,1,12,3,16,3,2,1,1,1,1,1,1,1,1,12,3,16,3,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0


## 7. Saving the Feature Dataset

The final client-level dataset contains the engineered features and the target variable.

The dataset is saved in Parquet format so that subsequent modeling notebooks can load the prepared data directly without repeating the feature engineering pipeline.

In [36]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DATA_DIR / "train_features.parquet"

In [37]:
final_dataset.to_parquet(FEATURES_PATH)

print(f"Saved to: {FEATURES_PATH}")

Saved to: C:\Study\SkillBox\data\processed\train_features.parquet
